# Aula 08 (Parte 2) — Ciência de Dados para ADS
## Laboratório Prático Ao Vivo: Variabilidade, Curva Normal, TLC e Tamanho de Amostra

Este notebook contém todos os exemplos práticos abordados nos slides da **Aula 08 (Parte 2)**.
Ele foi projetado para execução em sala de aula, permitindo compreender a medição da variabilidade através do **Desvio Padrão (SD)**, a **Curva Normal**, o **Teorema do Limite Central (TLC)**, a **Lei da Raiz Quadrada** e o cálculo do **Tamanho de Amostra** para pesquisas e sistemas.

---
### 1. Configuração do Ambiente
Importando `numpy`, `pandas`, `matplotlib`, `seaborn` e `scipy.stats`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
print("✅ Ambiente configurado para a Parte 2!")

---
### 2. Módulo 1: Calculando o Desvio Padrão (SD) e Unidades Padrão (Z-Score)
**Exemplo:** Coleção simples `any_numbers = [1, 2, 2, 10]`.

In [ ]:
any_numbers = np.array([1, 2, 2, 10])
media = np.mean(any_numbers)
desvios = any_numbers - media
desvios_quad = desvios ** 2
variancia = np.mean(desvios_quad)
sd = np.sqrt(variancia)

df_sd = pd.DataFrame({
    "x": any_numbers,
    "x - media": desvios,
    "(x - media)^2": desvios_quad,
    "Z-Score (Standard Units)": (any_numbers - media) / sd
})
display(df_sd)
print(f"Média: {media:.2f} | Variância: {variancia:.2f} | SD (np.std): {sd:.2f}")

---
### 3. Módulo 2: A Curva Normal e a Regra Empírica (68% - 95% - 99.7%)
Utilizando a função acumulada `scipy.stats.norm.cdf`.

In [ ]:
# Calculando probabilidades acumuladas sob a Curva Normal Padrao
area_1sd = stats.norm.cdf(1) - stats.norm.cdf(-1)
area_2sd = stats.norm.cdf(2) - stats.norm.cdf(-2)
area_3sd = stats.norm.cdf(3) - stats.norm.cdf(-3)

print(f"Área entre Media ± 1 SD: {area_1sd:.2%}")
print(f"Área entre Media ± 2 SD: {area_2sd:.2%}")
print(f"Área entre Media ± 3 SD: {area_3sd:.2%}")

In [ ]:
# Plotando a Curva Normal e os Intervalos de SD
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, y, color="#1E293B", linewidth=2.5, label="Normal Padrão N(0,1)")
ax.fill_between(x, y, where=(x >= -1) & (x <= 1), color="#0EA5E9", alpha=0.5, label="68% (±1 SD)")
ax.fill_between(x, y, where=(x >= -2) & (x <= 2), color="#0EA5E9", alpha=0.2, label="95% (±2 SD)")
ax.set_title("A Curva Normal Padrão e os Pontos de Inflexão (±1 SD)", fontsize=14, fontweight="bold")
ax.set_xlabel("Unidades Padrão (Z-Scores)")
ax.set_ylabel("Densidade")
ax.legend()
plt.show()

---
### 4. Módulo 3: O Teorema do Limite Central (TLC) na Prática
**Fenômeno:** Não importa quão assimétrica seja a população original (ex: atrasos de voos), a **distribuição das médias amostrais** de tamanho $N$ grande será sempre aproximada por uma Curva Normal!

In [ ]:
np.random.seed(42)
populacao = np.random.exponential(scale=15, size=10000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, n in enumerate([5, 25, 100]):
    amostras = np.random.choice(populacao, size=(10000, n))
    medias = np.mean(amostras, axis=1)
    sns.histplot(medias, bins=30, kde=True, ax=axes[idx], color="#0EA5E9", edgecolor="black")
    axes[idx].set_title(f"Médias Amostrais (N={n})", fontsize=12, fontweight="bold")
    axes[idx].set_xlabel(f"SD Observado: {np.std(medias):.2f}")

plt.tight_layout()
plt.show()

---
### 5. Módulo 4: A Lei da Raiz Quadrada e Acurácia
$$SD(\text{média amostral}) = \frac{SD_{\text{populacional}}}{\sqrt{n}}$$

In [ ]:
sd_pop = np.std(populacao)
sd_n25 = sd_pop / np.sqrt(25)
sd_n100 = sd_pop / np.sqrt(100)

print(f"SD da População Original:          {sd_pop:.2f}")
print(f"SD das Médias Amostrais (N=25):   {sd_n25:.2f}")
print(f"SD das Médias Amostrais (N=100):  {sd_n100:.2f}")
print(f"Aumento de precisão (Fator):      {sd_n25 / sd_n100:.2f}x (Multiplicar N por 4 divide SD por 2)")

---
### 6. Módulo 5: Tamanho de Amostra Mínimo em Pesquisas
**Cenário:** Estimar a intenção de voto do Candidato A com margem de erro total máxima de 1% (largura total do intervalo de 95% $\le 0.01$).


In [ ]:
sd_max = 0.5
largura_intervalo = 0.01
n_necessario = (4 * sd_max / largura_intervalo) ** 2
print(f"Tamanho mínimo de amostra necessário: {n_necessario:,.0f} pessoas")